# RPS Fine-Tuning — 바위(Rock) 인식 강화
DenseNet121 + class_weight로 바위 클래스 인식률을 높입니다.
- 클래스(class): 모델이 구분하는 카테고리 (0=가위, 1=바위, 2=보)
- class_weight: 바위를 틀렸을 때 손실을 2배로 반영 → 바위에 더 집중
- EarlyStopping으로 과적합 방지

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import os, pathlib, glob, cv2

print('TF:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
import pathlib

# 노트북 위치 기준 상대경로 (어떤 환경에서도 동작)
NOTEBOOK_DIR = pathlib.Path().resolve()
DATASET_DIR  = NOTEBOOK_DIR / 'RPS_Dataset'
RESULT_DIR   = NOTEBOOK_DIR / 'results'
RESULT_DIR.mkdir(exist_ok=True)

IMG_SIZE    = 64
BATCH_SIZE  = 32
CLASS_NAMES = ['scissors(가위)', 'rock(바위)', 'paper(보)']
print('데이터 경로:', DATASET_DIR)
print('결과 경로:', RESULT_DIR)


In [ ]:
%%time
def load_dataset(root):
    data, labels = [], []
    for label in range(3):
        for fpath in glob.glob(str(root / str(label) / '*.*')):
            img = cv2.imread(fpath)
            if img is None:
                continue
            img = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (IMG_SIZE, IMG_SIZE))
            data.append(img)
            labels.append(label)
    return np.array(data, dtype=np.float32), np.array(labels, dtype=np.int32)

all_data, all_labels = load_dataset(DATASET_DIR)
for i, name in enumerate(CLASS_NAMES):
    print(f'  label {i} {name}: {(all_labels==i).sum()}장')

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data, train_labels, test_labels = train_test_split(
    all_data, all_labels, test_size=0.2, random_state=42, stratify=all_labels)
print('train:', train_data.shape, '  test:', test_data.shape)

## 모델 빌드 (ImageNet 사전학습 + 커스텀 헤드)

In [ ]:
def build_model():
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = tf.keras.applications.densenet.preprocess_input(inputs)
    base = tf.keras.applications.DenseNet121(
        include_top=False, weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    x = base(x, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(3, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)

model = build_model()
print('총 파라미터:', model.count_params())
print('DenseNet 동결 상태 — ImageNet 가중치 사용')


## Fine-tuning 전 클래스별 성능 확인

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

y_pred_before = np.argmax(model.predict(test_data, verbose=0), axis=1)
print('=== Fine-tuning 전 클래스별 성능 ===')
print(classification_report(test_labels, y_pred_before, target_names=CLASS_NAMES))
print('혼동 행렬 (행=실제, 열=예측):')
print(confusion_matrix(test_labels, y_pred_before))

## 1단계: 헤드(분류기) 학습 — 바위 가중치 2배

In [ ]:
%%time
# class_weight: 바위(label=1)를 틀리면 손실이 2배 → 모델이 바위에 더 집중
class_weight = {0: 1.0, 1: 2.0, 2: 1.0}

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,
                                      restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                          patience=2, verbose=1)
]

# 1단계: DenseNet 동결 상태로 헤드만 빠르게 학습
history1 = model.fit(
    train_data, train_labels,
    epochs=10,
    batch_size=BATCH_SIZE,
    validation_data=(test_data, test_labels),
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1
)
print('1단계 완료')

## Fine-Tuning 후 클래스별 성능 비교

In [ ]:
from sklearn.metrics import recall_score, precision_score

y_pred_after = np.argmax(model.predict(test_data, verbose=0), axis=1)
print('=== Fine-tuning 후 클래스별 성능 ===')
print(classification_report(test_labels, y_pred_after, target_names=CLASS_NAMES))
print('혼동 행렬:')
print(confusion_matrix(test_labels, y_pred_after))

rock_recall_before = recall_score(test_labels, y_pred_before, average=None)[1]
rock_recall_after  = recall_score(test_labels, y_pred_after,  average=None)[1]
rock_prec_before   = precision_score(test_labels, y_pred_before, average=None)[1]
rock_prec_after    = precision_score(test_labels, y_pred_after,  average=None)[1]

print('\n=== 바위(Rock) 인식 변화 ===')
print(f'  재현율(Recall)  : {rock_recall_before:.4f} → {rock_recall_after:.4f}  ({rock_recall_after-rock_recall_before:+.4f})')
print(f'  정밀도(Precision): {rock_prec_before:.4f} → {rock_prec_after:.4f}  ({rock_prec_after-rock_prec_before:+.4f})')

In [ ]:
# 훈련 곡선 저장
all_loss     = history1.history['loss']    
all_val_loss = history1.history['val_loss']
all_acc      = history1.history['accuracy']    
all_val_acc  = history1.history['val_accuracy']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Fine-Tuning 훈련 곡선 (바위 강화)', fontsize=13, fontweight='bold')

axes[0].plot(all_loss,     label='Train Loss')
axes[0].plot(all_val_loss, label='Val Loss', linestyle='--')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot([x*100 for x in all_acc],     label='Train Acc')
axes[1].plot([x*100 for x in all_val_acc], label='Val Acc', linestyle='--')
axes[1].set_title('Accuracy (%)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
chart_path = str(RESULT_DIR / 'finetune_curves.png')
plt.savefig(chart_path, dpi=150, bbox_inches='tight')
plt.close()
print('차트 저장:', chart_path)

In [ ]:
# TFLite 변환 및 저장
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
tflite_path = str(RESULT_DIR / 'RPS_finetuned_rock.tflite')
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print('TFLite 저장:', tflite_path,
      f'({os.path.getsize(tflite_path)/1024/1024:.1f} MB)')